# Mini-project 2: Reliability of a Stay Cable in a Cable-Stayed Bridge

**Student Name:** Amilcar Diaz

**CCID:** adiazgar

## Summary

This notebook evaluates the reliability of a stay cable of a cable-stayed bridge in Alberta. The permanent tension of the cable is estimated from acceleration measurements using the taut-string (vibration) method, and the uncertainty in the cable properties and in the measured frequency is propagated to the tension and to a limit state function. The reliability index and the probability of failure are computed with the First-Order Second-Moment (FOSM) method and verified with Monte Carlo simulation.

## 1. Problem description

Stay cables carry the deck loads of a cable-stayed bridge to the pylons, so their tension is one of the most important quantities to monitor. If the tension is too high, the cable may approach its capacity; if it changes over time, it may indicate damage, loss of anchorage or changes in the structure.

A common and inexpensive way to estimate the tension of a cable in service is the **vibration method**:

1. An accelerometer is attached to the cable and records its ambient vibration (in this example, records of 200 s sampled at 50 Hz).
2. The Fast Fourier Transform (FFT) of the record is computed, and the fundamental frequency $f_1$ is obtained by averaging the five highest peaks.
3. The tension is computed with the taut-string formula, which relates the tension to the frequency, the mass per unit length and the length of the cable.

**Sources of uncertainty:** the tension is not measured directly but computed from quantities that are themselves uncertain:

- The **mass per unit length** $m$ (manufacturing tolerances of the strands, sheathing and grout).
- The **effective length** $L$ (the vibrating length depends on dampers and anchorage details).
- The **measured frequency** $f_1$ (frequency resolution, measurement noise and temperature effects).

In addition, the **resistance** of the cable $R$ and the **live load** $T_{LL}$ caused by traffic are also random variables.

## 2. Basic random variables

The cable is made of 37 strands of 15.7 mm, grade 1860 MPa (ASTM A416-16), with a minimum breaking strength of 279 kN per strand.

| Variable | Mean | CV | Distribution | Justification |
|---|---|---|---|---|
| $R$, resistance | 37 × 279 kN = 10 323 kN | 0.015 | Normal | Ali, Katsuchi and Yamada (2018); Nowak (1999) |
| $m$, total mass per unit length | 54 kg/m | 0.02 | Normal | Manufacturing tolerances (assumed) |
| $L$, effective length | 150 m | 0.03 | Normal | Uncertainty due to dampers and anchorages (assumed) |
| $f_1$, fundamental frequency ($n$ = 1) | 0.96 Hz | 0.01 | Normal | Hypothetical measurement; resolution, noise, temperature |
| $T_{LL}$, live load | 1.18 × nominal CL-800 tension | 0.18 | Lognormal | Pushka et al. (2022); Kulicki et al. (2007) |

**Independence:** all the variables are assumed to be statistically independent (no correlation). The resistance depends on the material, the mass on the manufacturing process, the length on the geometry, the frequency on the measurement and the live load on the traffic, so they come from different sources. For this reason, the FOSM variance does not include covariance terms:

$$\sigma_g^2 \approx \sum_i \left(\frac{\partial g}{\partial x_i}\right)^2 \sigma_{X_i}^2$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Resistance R (kN): 37 strands of 15.7 mm, grade 1860 MPa (ASTM A416-16)
n_strands = 37
R_strand = 279.0
mu_R = n_strands * R_strand
cv_R = 0.015
sigma_R = cv_R * mu_R

# Total mass per unit length m (kg/m)
mu_m = 54.0
cv_m = 0.02
sigma_m = cv_m * mu_m

# Effective length L (m)
mu_L = 150.0
cv_L = 0.03
sigma_L = cv_L * mu_L

# Fundamental frequency f1 (Hz)
n = 1
mu_f = 0.96
cv_f = 0.01
sigma_f = cv_f * mu_f

print(f"R:  mean = {mu_R:.1f} kN, std = {sigma_R:.1f} kN")
print(f"m:  mean = {mu_m:.2f} kg/m, std = {sigma_m:.3f} kg/m")
print(f"L:  mean = {mu_L:.1f} m, std = {sigma_L:.2f} m")
print(f"f1: mean = {mu_f:.3f} Hz, std = {sigma_f:.4f} Hz")

In [ ]:
# Live load T_LL (kN): lognormal, bias = 1.18, CV = 0.18
T_LL_nominal = None   # TODO: nominal CL-800 tension in the cable (kN)
bias_LL = 1.18
cv_LL = 0.18

mu_LL = bias_LL * T_LL_nominal
sigma_LL = cv_LL * mu_LL

# Parameters of the underlying normal distribution of ln(T_LL)
zeta_LL = np.sqrt(np.log(1 + cv_LL**2))
lambda_LL = np.log(mu_LL) - zeta_LL**2 / 2

print(f"T_LL: nominal = {T_LL_nominal:.1f} kN, mean = {mu_LL:.1f} kN, std = {sigma_LL:.1f} kN")
print(f"      lambda = {lambda_LL:.4f}, zeta = {zeta_LL:.4f}")

## 3. Quantities of interest

**Permanent tension** (taut-string formula, in N when $m$ is in kg/m, $L$ in m and $f_n$ in Hz):

$$T_{med} = 4\,m\,L^2\left(\frac{f_n}{n}\right)^2$$

**Limit state function** (failure if $g < 0$):

$$g = R - \left[\,4\,m\,L^2\left(\frac{f_n}{n}\right)^2 + T_{LL}\right]$$

## 4. Probabilistic analysis

### 4.1 FOSM: mean and standard deviation of $T_{med}$ and $g$

The function is linearized at the mean values. The partial derivatives of $T_{med}$ (with $n = 1$) are:

$$\frac{\partial T_{med}}{\partial m} = 4L^2f^2, \qquad \frac{\partial T_{med}}{\partial L} = 8\,m\,L\,f^2, \qquad \frac{\partial T_{med}}{\partial f} = 8\,m\,L^2f$$

and for $g$: $\dfrac{\partial g}{\partial R} = 1$, $\dfrac{\partial g}{\partial T_{LL}} = -1$, and $\dfrac{\partial g}{\partial x_i} = -\dfrac{\partial T_{med}}{\partial x_i}$ for $m$, $L$ and $f$.

In [ ]:
# FOSM for the permanent tension T_med (kN), evaluated at the mean values
mu_T = 4 * mu_m * mu_L**2 * (mu_f / n)**2 / 1000

dT_dm = 4 * mu_L**2 * (mu_f / n)**2 / 1000
dT_dL = 8 * mu_m * mu_L * (mu_f / n)**2 / 1000
dT_df = 8 * mu_m * mu_L**2 * mu_f / n**2 / 1000

var_T_m = (dT_dm * sigma_m)**2
var_T_L = (dT_dL * sigma_L)**2
var_T_f = (dT_df * sigma_f)**2
sigma_T = np.sqrt(var_T_m + var_T_L + var_T_f)

print(f"T_med: mean = {mu_T:.1f} kN, std = {sigma_T:.1f} kN, CV = {sigma_T / mu_T:.4f}")
print(f"T_med / mean resistance = {mu_T / mu_R:.3f}")
print("Contribution to the variance of T_med:")
print(f"   m: {var_T_m / sigma_T**2 * 100:.1f} %")
print(f"   L: {var_T_L / sigma_T**2 * 100:.1f} %")
print(f"   f: {var_T_f / sigma_T**2 * 100:.1f} %")

In [ ]:
# FOSM for the limit state function g (kN)
mu_g = mu_R - mu_T - mu_LL

dg_dR = 1
dg_dm = -dT_dm
dg_dL = -dT_dL
dg_df = -dT_df
dg_dLL = -1

sigma_g = np.sqrt((dg_dR * sigma_R)**2 + (dg_dm * sigma_m)**2 + (dg_dL * sigma_L)**2
                  + (dg_df * sigma_f)**2 + (dg_dLL * sigma_LL)**2)

print(f"g: mean = {mu_g:.1f} kN, std = {sigma_g:.1f} kN")

### 4.2 Reliability index and probability of failure

$$\beta = \frac{\mu_g}{\sigma_g}, \qquad P_f = \Phi(-\beta)$$

In [ ]:
beta_fosm = mu_g / sigma_g
Pf_fosm = norm.cdf(-beta_fosm)

print(f"FOSM: beta = {beta_fosm:.3f}, Pf = {Pf_fosm:.3e}")

### 4.3 Comparison with the target reliability index

- $\beta_T = 3.5$ (AASHTO LRFD, 75-year service life), consistent with the live load parameters, which come from the AASHTO calibration (Kulicki et al. 2007; Pushka et al. 2022).
- The corresponding acceptable probability of failure is $P_{f,T} = \Phi(-3.5) \approx 2.3 \times 10^{-4}$.
- If $\beta \geq \beta_T$, the cable satisfies the required safety level.

**Note:** $\beta_T = 3.5$ was calibrated for girders. A stay cable has a more brittle failure mode and more severe consequences, so a higher value could be required. As a reference, ISO 2394 requires 5.2 for brittle failure of class 1 structures (cited in Hou et al. 2020), although for a different reference period, so it is not directly comparable.

In [ ]:
beta_T = 3.5
Pf_T = norm.cdf(-beta_T)

print(f"Target: beta_T = {beta_T}, Pf_T = {Pf_T:.2e}")
if beta_fosm >= beta_T:
    print(f"beta = {beta_fosm:.2f} >= {beta_T}: the cable satisfies the target reliability.")
else:
    print(f"beta = {beta_fosm:.2f} < {beta_T}: the cable does NOT satisfy the target reliability.")

### 4.4 Monte Carlo simulation

The limit state function is nonlinear because $L$ and $f$ appear squared. FOSM linearizes $g$ at the mean values, so it is an approximation; Monte Carlo simulation does not linearize the function and is used to check how good the FOSM approximation is.

The effect of the lognormal variable ($T_{LL}$) is also evaluated, since $P_f = \Phi(-\beta)$ is only exact when $g$ is normal. For this, the simulation is repeated with $T_{LL}$ modeled as a normal variable with the same mean and standard deviation.

In [ ]:
# Monte Carlo samples of the basic random variables
N = 1_000_000
rng = np.random.default_rng(42)

R_s = rng.normal(mu_R, sigma_R, N)
m_s = rng.normal(mu_m, sigma_m, N)
L_s = rng.normal(mu_L, sigma_L, N)
f_s = rng.normal(mu_f, sigma_f, N)

# Permanent tension (kN)
T_s = 4 * m_s * L_s**2 * (f_s / n)**2 / 1000

print(f"T_med (Monte Carlo): mean = {T_s.mean():.1f} kN, std = {T_s.std():.1f} kN")
print(f"T_med (FOSM):        mean = {mu_T:.1f} kN, std = {sigma_T:.1f} kN")

# Histogram of T_med with the FOSM normal approximation
T_line = np.linspace(T_s.min(), T_s.max(), 300)

plt.figure(figsize=(8, 5))
plt.hist(T_s, bins=100, density=True, color="lightgray", edgecolor="white", label="Monte Carlo")
plt.plot(T_line, norm.pdf(T_line, mu_T, sigma_T), color="black", linewidth=2, label="FOSM (normal)")
plt.xlabel("Permanent tension T_med (kN)")
plt.ylabel("Probability density")
plt.title("Permanent tension estimated from the vibration method")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# Live load samples: lognormal (model) and normal (for comparison)
LL_s = rng.lognormal(lambda_LL, zeta_LL, N)
LL_normal_s = rng.normal(mu_LL, sigma_LL, N)

# Limit state function (kN)
g_s = R_s - T_s - LL_s
g_normal_s = R_s - T_s - LL_normal_s

n_fail = np.sum(g_s < 0)
n_fail_normal = np.sum(g_normal_s < 0)
Pf_mc = n_fail / N
Pf_mc_normal = n_fail_normal / N
beta_mc = g_s.mean() / g_s.std()

print(f"g (Monte Carlo): mean = {g_s.mean():.1f} kN, std = {g_s.std():.1f} kN, beta = {beta_mc:.3f}")
print(f"g (FOSM):        mean = {mu_g:.1f} kN, std = {sigma_g:.1f} kN, beta = {beta_fosm:.3f}")
print()
print(f"Pf FOSM                         = {Pf_fosm:.3e}")
if n_fail > 0:
    print(f"Pf Monte Carlo (T_LL lognormal) = {Pf_mc:.3e}  ({n_fail} failures)")
else:
    print(f"Pf Monte Carlo (T_LL lognormal) < {1 / N:.0e}  (0 failures in {N} samples)")
if n_fail_normal > 0:
    print(f"Pf Monte Carlo (T_LL normal)    = {Pf_mc_normal:.3e}  ({n_fail_normal} failures)")
else:
    print(f"Pf Monte Carlo (T_LL normal)    < {1 / N:.0e}  (0 failures in {N} samples)")

# Histogram of g with the FOSM normal approximation
g_line = np.linspace(min(g_s.min(), 0), g_s.max(), 300)

plt.figure(figsize=(8, 5))
plt.hist(g_s, bins=100, density=True, color="lightgray", edgecolor="white", label="Monte Carlo")
plt.plot(g_line, norm.pdf(g_line, mu_g, sigma_g), color="black", linewidth=2, label="FOSM (normal)")
plt.axvline(0, color="red", linestyle="--", label="Failure (g = 0)")
plt.xlabel("Limit state function g (kN)")
plt.ylabel("Probability density")
plt.title("Distribution of the limit state function")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### 4.5 Linear correlation between $g$ and the basic random variables

With FOSM, the correlation coefficient between $g$ and each variable is approximated by:

$$\rho_{g,X_i} \approx \frac{\frac{\partial g}{\partial x_i}\,\sigma_{X_i}}{\sigma_g}$$

These values are compared with the sample correlation coefficients from the Monte Carlo simulation.

In [ ]:
# FOSM correlation coefficients
rho_R_fosm = dg_dR * sigma_R / sigma_g
rho_m_fosm = dg_dm * sigma_m / sigma_g
rho_L_fosm = dg_dL * sigma_L / sigma_g
rho_f_fosm = dg_df * sigma_f / sigma_g
rho_LL_fosm = dg_dLL * sigma_LL / sigma_g

# Monte Carlo sample correlation coefficients
rho_R_mc = np.corrcoef(g_s, R_s)[0, 1]
rho_m_mc = np.corrcoef(g_s, m_s)[0, 1]
rho_L_mc = np.corrcoef(g_s, L_s)[0, 1]
rho_f_mc = np.corrcoef(g_s, f_s)[0, 1]
rho_LL_mc = np.corrcoef(g_s, LL_s)[0, 1]

print("Correlation with g:      FOSM     Monte Carlo")
print(f"   R                 {rho_R_fosm:8.3f}   {rho_R_mc:8.3f}")
print(f"   m                 {rho_m_fosm:8.3f}   {rho_m_mc:8.3f}")
print(f"   L                 {rho_L_fosm:8.3f}   {rho_L_mc:8.3f}")
print(f"   f1                {rho_f_fosm:8.3f}   {rho_f_mc:8.3f}")
print(f"   T_LL              {rho_LL_fosm:8.3f}   {rho_LL_mc:8.3f}")

# Bar chart
names = ["R", "m", "L", "f1", "T_LL"]
rho_fosm = [rho_R_fosm, rho_m_fosm, rho_L_fosm, rho_f_fosm, rho_LL_fosm]
rho_mc = [rho_R_mc, rho_m_mc, rho_L_mc, rho_f_mc, rho_LL_mc]
x_pos = np.arange(len(names))

plt.figure(figsize=(8, 5))
plt.bar(x_pos - 0.2, rho_fosm, width=0.4, color="black", label="FOSM")
plt.bar(x_pos + 0.2, rho_mc, width=0.4, color="gray", label="Monte Carlo")
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(x_pos, names)
plt.ylabel("Correlation coefficient with g")
plt.title("Linear correlation between g and the basic random variables")
plt.grid(alpha=0.3, axis="y")
plt.legend()
plt.show()

## 5. Research paper summary

**Hou, Sun and Chen (2020).** *Journal of Bridge Engineering*, 25(7). DOI: [10.1061/(ASCE)BE.1943-5592.0001573](https://doi.org/10.1061/(ASCE)BE.1943-5592.0001573)

- The paper uses the same vibration method to estimate cable tensions and the same reliability index used in this notebook.
- **Contribution of the paper:** cable tensions measured during one year of monitoring.
- **Contribution of this example:** propagation of the uncertainty in $m$, $L$ and $f$ to the estimated tension and to the reliability of the cable.

**What I understood:**

*(To be written by the author.)*

**What remains unclear:**

*(To be written by the author.)*

## 6. Assumptions and limitations

- The basic random variables are assumed to be statistically independent.
- FOSM is a linear approximation of a nonlinear function.
- The target reliability index of 3.5 is taken from AASHTO (calibrated for girders); a higher value could be required for stay cables.
- The mean resistance is taken as the minimum specified breaking strength, which gives a conservative result.
- The live load parameters come from another study (calibrated with the HL-93 load model) and are applied to the CL-800 load only for this example.
- The measured frequency and the coefficients of variation of $m$, $L$ and $f$ are illustrative.
- The taut-string formula neglects the bending stiffness of the cable, the sag (catenary effect) and the effect of the dampers.
- The reliability index corresponds to a 75-year service life.

## 7. References

- Ali, Katsuchi and Yamada (2018). *Journal of Structural Engineering, JSCE*, 64A: 99–108. DOI: 10.11532/structcivil.64A.99
- ASTM A416/A416M-16. *Standard Specification for Low-Relaxation, Seven-Wire Steel Strand for Prestressed Concrete.*
- CSA S6. *Canadian Highway Bridge Design Code.*
- Hou, Sun and Chen (2020). *Journal of Bridge Engineering*, 25(7). DOI: 10.1061/(ASCE)BE.1943-5592.0001573
- ISO 2394:2015. *General principles on reliability for structures.*
- Kulicki et al. (2007). NCHRP 20-7.
- Nowak (1999). NCHRP Report 368.
- Pushka et al. (2022). *Canadian Journal of Civil Engineering*, 49(6): 899–909. DOI: 10.1139/cjce-2021-0031